In [1]:
%run forward_simulation.ipynb

# Code generation

In [ ]:
# Use export_vehicle(new_output_directory, cuda=True) below to export all vehicle consumers together.

In [3]:
# xnext.casadi is exported with the plant and EKF by export_vehicle below.


In [11]:
F_thrusters = uv_body.control_Allocation()
F_thrusters_fun = ca.Function("F_thrusters", [Tc, tau_b], [F_thrusters])

# F_thrusters_fun.generate("F_thrust.c")
# os.system(f"gcc -fPIC -shared F_thrust.c -o libF_thrust.so")

In [12]:
F_thruster_body = uv_body.thruster_input2generalized_Forces()
F_thruster_body_fun = ca.Function("F_thruster_body", [Tc, thru_u], [F_thruster_body])

F_thruster_body_fun.generate("F_thruster_body.c")
os.system(f"gcc -fPIC -shared F_thruster_body.c -o libF_thruster_body.so")

0

In [13]:
F_thrusters_fun(blue.thrust_config, [0.5,0,0,0,0,0])

DM([0.176803, 0.176803, -0.176803, -0.176803, -0.0884017, 0.0884017, -0.0884017, 0.0884017])

## Canonical submerged-vehicle export
Run the next cell, then call export_vehicle(new_output_directory, cuda=True).
This is the deployment export entry point; other notebooks demonstrate models.
The same source and integration settings generate plant, EKF, controllers and CPU/CUDA kernels.

In [ ]:
# Canonical vehicle export: plant, EKF, control models and optional batch kernels.
def export_vehicle(output, *, cuda=False, native=True, compiler="gcc", optimization="-O1",
                   artificial_surface_support=False):
    """Export one consistently configured vehicle model; never installs into a live workspace."""
    import os, json, hashlib, subprocess
    from pathlib import Path
    import casadi as ca
    from diffUV.base import Base
    from diffUV.simulator import Simulator
    from diffUV.controller import Controller
    from diffUV.kinematics import Kinematics
    from diffUV.utils.symbols import n, x_nb, v_c, eul, Kp, Ki, Kd, sum_e_buffer, nd, xb_d, dt, des_acc_b
    from diffUV.estimation import (disturbance_ekf_functions, disturbance_ekf_correction,
        vehicle_sensor_model, disturbance_ekf_sensor_correction,
        disturbance_ekf_covariance_prediction, disturbance_ekf_smoothing_moment,
        disturbance_ekf_smoothing_full_moment)
    from diffUV.identification_data import IntegralVehicleRegressor
    output=Path(output).resolve()
    output.mkdir(parents=True, exist_ok=False)
    previous=Path.cwd()
    model=Base(use_jit=False, artificial_surface_support=artificial_surface_support)
    step=Simulator(model).model_reg()
    plant=ca.Function('Vnext_reg', list(step[1:]), [step[0]])
    if native:
        predict, update=disturbance_ekf_functions(artificial_surface_support=artificial_surface_support)
        observer=(predict,update,disturbance_ekf_correction(),vehicle_sensor_model(artificial_surface_support=artificial_surface_support),
            disturbance_ekf_sensor_correction(artificial_surface_support=artificial_surface_support),disturbance_ekf_covariance_prediction(predict),
            disturbance_ekf_smoothing_moment(),disturbance_ekf_smoothing_full_moment())
    from diffUV.utils.rotation_error import rotation_log
    rotation=ca.SX.sym("rotation",3,3)
    control=Controller(model)
    args=[control.lumped_params,Kp,Ki,Kd,sum_e_buffer,ca.vertcat(n,x_nb),nd,xb_d]
    functions={
      'xnext.casadi':plant,
      'rotation_log.casadi':ca.Function('rotation_log',[rotation],[rotation_log(rotation)]),
      'uv_reg_pid_controller.casadi':ca.Function('pid_control_reg',args+[dt],list(control.position_pid_reg())),
      'uv_trackingController.casadi':ca.Function('trackingpid',args+[des_acc_b,dt,v_c],list(control.trajectorytracking_pid()))}
    smc_gains=[ca.SX.sym(name,6) for name in
               ('surface_gain','damping_gain','switching_gain','boundary')]
    functions['uv_sliding_mode_controller.casadi']=ca.Function('sliding_mode_control',
        [control.lumped_params,ca.vertcat(n,x_nb),nd,xb_d,des_acc_b]+smc_gains,
        list(control.sliding_mode(*smc_gains)),
        ['parameters','state','desired_pose','desired_twist','desired_acceleration',
         'surface_gain','damping_gain','switching_gain','boundary'],
        ['wrench','sliding','pose_error'])
    _,_,_,_,p,M,C,g,D=model.build_sys_regressor()
    params=ca.vertcat(p,v_c)
    # g_id retains its legacy attitude-only ABI (submerged restoring matrix).
    # Depth-dependent consumers must use g_depth_id when surface support is on.
    functions.update({
      'M_id.casadi':ca.Function('M_b_id',[params],[M]),
      'C_id.casadi':ca.Function('C_id',[x_nb,params],[C]),
      'g_id.casadi':ca.Function('g_id',[eul,params],[g]),
      'g_depth_id.casadi':ca.Function('g_depth_id',[n,params],[model.surface_restoring_reg()],
                          ['pose_ned','model_parameters'],['restoring_force_body']),
      'body_damping_matrix_id.casadi':ca.Function('body_damping_matrix_id',[x_nb,params],[D]),
      'J_uv.casadi':ca.Function('J_uv',[eul],[Kinematics().J])})
    try:
        os.chdir(output)
        for filename,function in functions.items(): function.save(filename)
        if native:
            for unit,library,group in [('Vnext','libUV_xnext.so',(plant,)),
                                        ('disturbance_ekf','libdisturbanceEKF_next.so',observer),
                                        ('sliding_mode','libsliding_mode_controller.so',
                                         (functions['uv_sliding_mode_controller.casadi'],))]:
                generator=ca.CodeGenerator(unit+'.c',{'with_header':True})
                for function in group:generator.add(function)
                generator.generate()
                subprocess.run([compiler,optimization,'-fPIC','-shared',unit+'.c','-o',library,'-lm'],check=True)
        regressor=IntegralVehicleRegressor(artificial_surface_support=artificial_surface_support)
        generator=ca.CodeGenerator('vehicle_regressor.c',{'with_header':True})
        for original in (regressor.inertia,regressor.bias,regressor.weights):
            inputs=[ca.MX.sym(original.name_in(i),*original.size_in(i)) for i in range(original.n_in())]
            generator.add(ca.Function(original.name(),inputs,[ca.densify(original(*inputs))]))
        generator.generate()
        if cuda:
            for name in ('Vnext_reg','Vnext_reg_gpu'):
                function=ca.Function(name,list(step[1:]),[step[0]])
                generator=ca.CodeGenerator(name,{'with_header':True,'casadi_real':'double',
                    'cpp':False,'cuda':True,'cuda_kernels':{name:{
                        'batch_inputs':list(range(5)),'kernel_name':name+'_kernel',
                        'device_name':'device_'+name+'_eval'}}})
                generator.add(function);generator.generate()
        import diffUV
        root=Path(diffUV.__file__).resolve().parent
        manifest={'model':('historical artificial surface support' if artificial_surface_support
                   else 'fully submerged; no air/water interface'),
          'artificial_surface_support': artificial_surface_support,
          'native':native, 'cuda':cuda, 'compiler':compiler,'compiler_flags':[optimization,'-fPIC','-shared'],
          'casadi_version':ca.__version__,'frame':'NED / body FRD',
          'source_sha256':{str(f.relative_to(root)):hashlib.sha256(f.read_bytes()).hexdigest()
                           for f in root.rglob('*.py')},
          'artifact_sha256':{f.name:hashlib.sha256(f.read_bytes()).hexdigest()
                            for f in output.iterdir() if f.is_file()}}
        (output/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    finally:
        os.chdir(previous)
    return output
